# 🌊 Nori's Deep Ocean Mood Engine v2
### A Data-Driven Pet Simulator for Ocean Education

---

## What this notebook teaches

Nori is Argo float **4901639** — an autonomous robot drifting through the Pacific Ocean. She dives from the sunlit surface all the way to 2000m depth, collecting temperature, salinity, and pressure readings every ~10 days.

This notebook turns that raw data into a **pet simulator**: Nori has moods, and her moods change depending on where she is in the ocean and what the conditions are like. Each mood maps to a real ecological consequence.

---

## The Ocean Has Layers — and Each One Has Different Rules

| Zone | Depth | What lives here | Why it matters |
|---|---|---|---|
| 🌞 Sunlight Zone | 0–200m | Coral, fish, sea turtles, plankton | Most marine life lives here. Warm anomalies cause bleaching. |
| 🌅 Twilight Zone | 200–1000m | Thermocline, bioluminescent creatures, oxygen minimum | Heat penetrating here is a serious climate signal |
| 🌑 Midnight Zone | 1000–2000m | Deep sea creatures, pitch black, high pressure | Takes centuries to warm naturally — any anomaly is alarming |

---

## Two Factors Drive Nori's Mood
1. **Temperature** — is it warmer or colder than what's normal *at this depth*?
2. **Salinity** — is the water fresher or saltier than normal? (Freshwater = glacial melt. Too salty = evaporation anomaly.)

Both are measured against **baselines computed from Nori's own 168 cycles** — not external references.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
COLS = [
    'pressure (decibar)',
    'temperature (degree_celsius)',
    'salinity (dimensionless)',
    'meta_cycle_number'
]

df = pd.read_csv('single_argo.csv', usecols=COLS)
df = df.dropna(subset=['temperature (degree_celsius)'])

print(f"Loaded {len(df):,} rows across {df['meta_cycle_number'].nunique()} cycles")
print(f"Pressure: {df['pressure (decibar)'].min()}–{df['pressure (decibar)'].max()} decibar")
print(f"Temperature: {df['temperature (degree_celsius)'].min():.2f}–{df['temperature (degree_celsius)'].max():.2f} °C")
print(f"Salinity: {df['salinity (dimensionless)'].min():.2f}–{df['salinity (dimensionless)'].max():.2f}")
df.head()

## 2. Classify Depth Zones
Each reading is assigned to a depth zone based on pressure (1 decibar ≈ 1 meter depth).

In [ ]:
def get_depth_zone(pressure):
    if pressure < 200:
        return 'Sunlight Zone 🌞'
    elif pressure < 1000:
        return 'Twilight Zone 🌅'
    else:
        return 'Midnight Zone 🌑'

df['Zone'] = df['pressure (decibar)'].apply(get_depth_zone)

print("Reading count per zone:")
print(df['Zone'].value_counts())

## 3. Compute Real Baselines from Nori's Own Data
Rather than using external reference values, we compute what is **normal for Nori at each depth zone** directly from her 168 cycles. This avoids geographic bias.

In [ ]:
zone_stats = df.groupby('Zone').agg(
    temp_mean=('temperature (degree_celsius)', 'mean'),
    temp_std =('temperature (degree_celsius)', 'std'),
    sal_mean =('salinity (dimensionless)',      'mean'),
    sal_std  =('salinity (dimensionless)',      'std'),
    count    =('temperature (degree_celsius)', 'count')
).round(3)

# Tolerance = 0.5 standard deviations — Nori's comfort zone at each depth
zone_stats['temp_tolerance'] = (zone_stats['temp_std'] * 0.5).round(3)
zone_stats['sal_tolerance']  = (zone_stats['sal_std']  * 0.5).round(3)

print("Nori's data-derived baselines per depth zone:")
zone_stats

## 4. Depth-Aware Mood Engine

Nori's mood is determined by **two factors**:
- **Temperature status** — warm / cold / normal *relative to her zone baseline*
- **Salinity status** — fresh / salty / normal *relative to her zone baseline*

The mood and its ecological consequence change depending on which zone she is in.

In [ ]:
ZONE_MOODS = {
    'Sunlight Zone 🌞': {
        'warm':  ('Sweating 🥵',   'Coral bleaching risk. Fish migrating to cooler water. Sea turtles disoriented.'),
        'cold':  ('Shivering 🥶',  'Cold upwelling bringing nutrients up. Algae blooming. Great for fish, hard on coral.'),
        'happy': ('Thriving 🐠',   'Surface ecosystem in balance. Coral healthy. Fish and plankton thriving.'),
    },
    'Twilight Zone 🌅': {
        'warm':  ('Stressed 😰',   'Heat penetrating deeper than normal. Oxygen minimum zone expanding. Bioluminescent creatures displaced.'),
        'cold':  ('Hiding 🫧',     'Cold layer thickening. Deep species pushed upward. Thermocline sharpening.'),
        'happy': ('Exploring 🔦',  'Thermocline stable. Twilight creatures undisturbed. Oxygen levels normal.'),
    },
    'Midnight Zone 🌑': {
        'warm':  ('Alarmed 🚨',    'Deep ocean warming detected. This takes centuries naturally — a serious long-term climate signal.'),
        'cold':  ('Dormant 🧊',    'Dense cold water sinking from polar regions. Thermohaline circulation active.'),
        'happy': ('Resting 🌑',    'Deep ocean stable. Ancient water conditions preserved. High pressure ecosystem undisturbed.'),
    }
}

SALINITY_STATUS = {
    'fresh': '💧 Too Fresh — possible glacial melt or heavy rainfall. Disrupts density-driven circulation.',
    'salty': '🧂 Too Salty — evaporation exceeding rainfall, or circulation anomaly. Ocean getting denser.',
    'normal': '✅ Salinity Normal'
}

def assign_mood(row):
    zone = row['Zone']
    temp = row['temperature (degree_celsius)']
    sal  = row['salinity (dimensionless)']
    z    = zone_stats.loc[zone]

    temp_anomaly = temp - z['temp_mean']
    sal_anomaly  = sal  - z['sal_mean']  if pd.notna(sal) else 0

    # Temperature status
    if abs(temp_anomaly) <= z['temp_tolerance']:
        temp_status = 'happy'
    elif temp_anomaly > 0:
        temp_status = 'warm'
    else:
        temp_status = 'cold'

    # Salinity status
    if abs(sal_anomaly) <= z['sal_tolerance']:
        sal_status = 'normal'
    elif sal_anomaly > 0:
        sal_status = 'salty'
    else:
        sal_status = 'fresh'

    mood, consequence = ZONE_MOODS[zone][temp_status]
    sal_note = SALINITY_STATUS[sal_status]

    return pd.Series({
        'Temp Anomaly':    round(temp_anomaly, 3),
        'Sal Anomaly':     round(sal_anomaly,  3),
        'Temp Status':     temp_status,
        'Sal Status':      sal_status,
        'Mood':            mood,
        'Consequence':     consequence,
        'Salinity Note':   sal_note
    })

mood_df = df.apply(assign_mood, axis=1)
df = pd.concat([df, mood_df], axis=1)

print("Mood assigned to all readings.")
df[['pressure (decibar)', 'Zone', 'temperature (degree_celsius)', 'Temp Anomaly',
    'Mood', 'salinity (dimensionless)', 'Sal Status']].head(10)

## 5. Cycle × Zone Statistical Summary
For each cycle and each zone, we run a t-test to determine if the temperature anomaly is statistically significant (p < 0.05).

In [ ]:
def get_cycle_zone_status(group):
    zone = group['Zone'].iloc[0]
    temps = group['temperature (degree_celsius)'].dropna()
    if len(temps) < 2:
        return None

    z            = zone_stats.loc[zone]
    baseline     = z['temp_mean']
    tolerance    = z['temp_tolerance']
    mean_temp    = temps.mean()
    anomaly      = mean_temp - baseline
    _, p_value   = stats.ttest_1samp(temps, baseline)

    if abs(anomaly) <= tolerance:
        mood, confidence = ZONE_MOODS[zone]['happy'][0], 'Within Comfort Zone'
    elif anomaly > tolerance and p_value < 0.05:
        mood, confidence = ZONE_MOODS[zone]['warm'][0],  'Statistically Significant'
    elif anomaly < -tolerance and p_value < 0.05:
        mood, confidence = ZONE_MOODS[zone]['cold'][0],  'Statistically Significant'
    else:
        mood, confidence = ZONE_MOODS[zone]['happy'][0], 'Normal/Inconclusive'

    decision = {'warm': '🤿 Dive', 'cold': '🆙 Surface', 'happy': '✅ Maintain'}
    temp_status = 'warm' if anomaly > tolerance else ('cold' if anomaly < -tolerance else 'happy')

    return pd.Series({
        'Zone':            zone,
        'baseline':        round(baseline, 3),
        'mean_temp':       round(mean_temp, 3),
        'anomaly':         round(anomaly, 3),
        'p_value':         round(p_value, 4),
        'Mood':            mood,
        'Confidence':      confidence,
        'Decision':        decision[temp_status]
    })

summary = (
    df.groupby(['meta_cycle_number', 'Zone'])
    .apply(get_cycle_zone_status)
    .dropna()
    .reset_index(level=2, drop=True)
    .reset_index()
)

print(f"Summary: {len(summary)} cycle × zone combinations")
summary.head(12)

## 6. Ocean Health Consequences
Every anomalous cycle gets a severity rating and a plain-English consequence — the bridge between Nori's mood and real ecosystem impact.

In [ ]:
CONSEQUENCES = {
    'Sunlight Zone 🌞': {
        'warm': {
            3.0: ('🚨 Critical', 'Coral reefs, sea turtles, tuna',
                  'Mass bleaching imminent. Hypoxic zones forming. Fish evacuating.'),
            2.0: ('⚠️ High',     'Coral reefs, jellyfish, seabirds',
                  'Bleaching risk rising. Jellyfish blooms expanding. Seabird food chain disrupted.'),
            0.0: ('⚡ Moderate', 'Coral reefs, tropical fish',
                  'Early coral stress. Tropical fish beginning to migrate northward.')
        },
        'cold': {
            3.0: ('⚠️ High',     'Warm-water fish, coral',
                  'Strong cold upwelling. Nutrient surge. Warm-water species displaced.'),
            0.0: ('⚡ Moderate', 'Some tropical species',
                  'Cold upwelling boosting plankton. Mild disruption to surface species.')
        }
    },
    'Twilight Zone 🌅': {
        'warm': {
            2.0: ('🚨 Critical', 'Bioluminescent creatures, oxygen-dependent species',
                  'Heat invading twilight zone. Oxygen minimum zone expanding rapidly.'),
            0.0: ('⚠️ High',     'Mesopelagic fish, squid',
                  'Thermocline shifting. Deep species losing their habitat boundary.')
        },
        'cold': {
            0.0: ('⚡ Moderate', 'Twilight zone creatures',
                  'Cold layer deepening. Species being pushed toward the surface.')
        }
    },
    'Midnight Zone 🌑': {
        'warm': {
            0.5: ('🚨 Critical', 'Deep sea ecosystems, thermohaline circulation',
                  'Deep ocean warming — this takes centuries naturally. Major long-term climate signal.'),
            0.0: ('⚠️ High',     'Deep sea organisms',
                  'Abnormal warmth at depth. Deep sea species have no escape route.')
        },
        'cold': {
            0.0: ('🔵 Low',      'Thermohaline circulation',
                  'Dense cold water sinking. Could intensify deep ocean circulation.')
        }
    }
}

def get_severity(zone, temp_status, anomaly):
    if temp_status == 'happy':
        return pd.Series({'Severity': '✅ Normal', 'At Risk': 'None',
                          'Health Consequence': 'Ecosystem stable at this depth.'})
    thresholds = CONSEQUENCES.get(zone, {}).get(temp_status, {})
    abs_a = abs(anomaly)
    for threshold in sorted(thresholds.keys(), reverse=True):
        if abs_a >= threshold:
            sev, at_risk, consequence = thresholds[threshold]
            return pd.Series({'Severity': sev, 'At Risk': at_risk,
                              'Health Consequence': consequence})
    return pd.Series({'Severity': '✅ Normal', 'At Risk': 'None',
                      'Health Consequence': 'Conditions within normal range.'})

summary['temp_status'] = summary.apply(
    lambda r: 'warm' if r['anomaly'] > zone_stats.loc[r['Zone'], 'temp_tolerance']
              else ('cold' if r['anomaly'] < -zone_stats.loc[r['Zone'], 'temp_tolerance'] else 'happy'),
    axis=1
)

health = summary.apply(
    lambda r: get_severity(r['Zone'], r['temp_status'], r['anomaly']), axis=1
)
summary = pd.concat([summary, health], axis=1)

# Show only anomalous cycles
anomalous = summary[summary['Mood'] != summary.apply(
    lambda r: ZONE_MOODS[r['Zone']]['happy'][0], axis=1
)][['meta_cycle_number', 'Zone', 'anomaly', 'Mood', 'Decision', 'Severity', 'At Risk', 'Health Consequence']]

print(f"{len(anomalous)} anomalous cycle × zone combinations:")
anomalous.head(20)

## 7. Visualization — Nori's Mood Across All Depths
Each dot is one reading. Color = Mood. Y-axis is inverted so surface is at top and deep ocean at bottom.

In [ ]:
MOOD_COLORS = {
    'Sweating 🥵':   '#e74c3c',
    'Shivering 🥶':  '#3498db',
    'Thriving 🐠':   '#2ecc71',
    'Stressed 😰':   '#e67e22',
    'Hiding 🫧':     '#9b59b6',
    'Exploring 🔦':  '#1abc9c',
    'Alarmed 🚨':    '#c0392b',
    'Dormant 🧊':    '#85c1e9',
    'Resting 🌑':    '#566573',
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# --- Left plot: Temperature vs Depth colored by Mood ---
for mood, color in MOOD_COLORS.items():
    mask = df['Mood'] == mood
    if mask.sum() > 0:
        ax1.scatter(
            df.loc[mask, 'temperature (degree_celsius)'],
            df.loc[mask, 'pressure (decibar)'],
            c=color, alpha=0.4, s=8, label=mood
        )

# Zone baseline lines
for zone, row in zone_stats.iterrows():
    ax1.axvline(x=row['temp_mean'], linewidth=1.2, linestyle='--', alpha=0.7,
                label=f"{zone} baseline ({row['temp_mean']:.1f}°C)")

# Zone depth boundaries
ax1.axhline(y=200,  color='gray', linewidth=0.8, linestyle=':', alpha=0.6)
ax1.axhline(y=1000, color='gray', linewidth=0.8, linestyle=':', alpha=0.6)
ax1.text(2.5, 100,  '🌞 Sunlight',  fontsize=8, color='gray')
ax1.text(2.5, 600,  '🌅 Twilight',  fontsize=8, color='gray')
ax1.text(2.5, 1500, '🌑 Midnight',  fontsize=8, color='gray')

ax1.invert_yaxis()
ax1.set_xlabel('Temperature (°C)', fontsize=12)
ax1.set_ylabel('Pressure (decibar) — deeper ↓', fontsize=12)
ax1.set_title("Nori's Mood vs. Depth\nEach dot = one reading", fontsize=13)
ax1.legend(fontsize=7, loc='lower right')
ax1.grid(True, alpha=0.2)

# --- Right plot: Mood distribution per zone ---
zone_mood_counts = df.groupby(['Zone', 'Mood']).size().unstack(fill_value=0)
zone_order = ['Sunlight Zone 🌞', 'Twilight Zone 🌅', 'Midnight Zone 🌑']
zone_mood_counts = zone_mood_counts.reindex(zone_order)

colors = [MOOD_COLORS.get(m, '#aaa') for m in zone_mood_counts.columns]
zone_mood_counts.plot(kind='bar', ax=ax2, color=colors, edgecolor='white', linewidth=0.5)

ax2.set_title("Mood Distribution per Zone\nHow often is Nori stressed at each depth?", fontsize=13)
ax2.set_xlabel('Depth Zone', fontsize=11)
ax2.set_ylabel('Number of Readings', fontsize=11)
ax2.set_xticklabels(['Sunlight\n🌞', 'Twilight\n🌅', 'Midnight\n🌑'], rotation=0)
ax2.legend(fontsize=7, loc='upper right')
ax2.grid(True, alpha=0.2, axis='y')

plt.suptitle("Nori's Deep Ocean Mood Engine v2", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('nori_mood_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved as nori_mood_v2.png")

## 8. Salinity Health Check
Temperature is only half the story. Salinity tells us about freshwater influx (glacier melt), evaporation rates, and ocean circulation.

In [ ]:
sal_summary = df.groupby(['Zone', 'Sal Status']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
sal_colors = {'fresh': '#85c1e9', 'normal': '#2ecc71', 'salty': '#e59866'}
colors = [sal_colors.get(c, '#aaa') for c in sal_summary.columns]
sal_summary.reindex(zone_order).plot(kind='bar', ax=ax, color=colors, edgecolor='white')

ax.set_title("Salinity Status per Zone\n💧 Fresh = glacial melt risk | 🧂 Salty = evaporation anomaly", fontsize=13)
ax.set_xlabel('Depth Zone')
ax.set_ylabel('Number of Readings')
ax.set_xticklabels(['Sunlight\n🌞', 'Twilight\n🌅', 'Midnight\n🌑'], rotation=0)
ax.legend(['💧 Fresh', '✅ Normal', '🧂 Salty'], fontsize=10)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.savefig('nori_salinity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSalinity distribution:")
print(df['Sal Status'].value_counts())

## 9. Full Mood Distribution Summary

In [ ]:
print("=== Row-level Mood Distribution (all readings) ===")
print(df['Mood'].value_counts())
print()
print("=== Cycle × Zone Mood Distribution (statistical) ===")
print(summary['Mood'].value_counts())
print()
print("=== Decision Distribution ===")
print(summary['Decision'].value_counts())
print()
print("=== Severity Distribution ===")
print(summary['Severity'].value_counts())

## 🎓 What Students Learn From This

| What they see | What it teaches |
|---|---|
| Nori's mood changes with depth | The ocean has distinct layers — each with different physics and ecosystems |
| Sunlight zone is often Sweating 🥵 | Surface warming is a real, measurable phenomenon |
| Midnight zone anomalies are rare but alarming | Deep ocean takes centuries to change — any signal is serious |
| Salinity anomalies | Freshwater influx from glaciers and ice melt is detectable in ocean data |
| Dive/Surface/Maintain decisions | Autonomous robots make data-driven decisions just like a doctor reading vitals |
| p-value < 0.05 | Statistical significance separates real signals from random noise |
| Baselines from Nori's own data | Good science uses location-appropriate references, not generic values |

> **The big idea:** Nori isn't just a robot checking her own comfort. She is an early warning system for the entire ocean ecosystem. When she is Alarmed 🚨 in the Midnight Zone, marine life two kilometers above her is already at risk.